# Memory Notebook - Samuel Robson
**Course:** INFO 290: Fundamentals of Generative AI (Spring 2026)  
**Purpose:** Testing each component of what will be `memory.py` in our implementation
#### ``Why this matters``
The memory stream is the core of the Generative Agents architecture. Every observation, decision, reflection, and conversation gets stored here. In Stage 2, `retrieval.py` will search this stream to find the most relevant memories at decision time.

---

## 1. Setup & Imports

In [ ]:
import numpy as np
from dataclasses import dataclass, field
from typing import List, Optional

# ── Memory types ──────────────────────────────────────────────────────────────
# the four types of memories an agent can have — used for validation in Memory.__post_init__()
# observation:  something that happened to the agent (e.g. received an insurance letter)
# decision:     something the agent decided to do (e.g. called their Firewise group)
# reflection:   a higher-level belief synthesised from multiple memories (Stage 5)
# conversation: something said in a multi-agent dialogue (Stage 4)
MEMORY_TYPES = {"observation", "decision", "reflection", "conversation"}


# The dataclass decorator automatically generates __init__ so we don't have to write:
#   def __init__(self, timestamp, description, importance, embedding, type): ...
# Python creates it for us based on the field declarations below
@dataclass
class Memory:
    """A single entry in an agent's memory stream."""

    # the simulation day this memory was created
    # day 0 = seed memories (before simulation starts)
    timestamp: int

    # the memory written in first-person natural language
    # e.g. 'I observed the fire shift direction to the north'
    description: str

    # how important this memory is on a scale of 1-10
    # scored by Claude via score_importance() in client.py
    # 1 = mundane (routine dispatch), 10 = life-altering (crew in danger)
    importance: int

    # the description converted to a 384-dimensional vector by all-MiniLM-L6-v2
    # used in Stage 2 retrieval to find memories semantically similar to a new situation
    embedding: np.ndarray

    # one of: 'observation', 'decision', 'reflection', 'conversation'
    type: str

    def __post_init__(self):
        # __post_init__ runs automatically after __init__ — used for validation
        # raises an error immediately if someone tries to create a Memory with an invalid type
        # or an importance score outside 1-10, catching bugs early
        if self.type not in MEMORY_TYPES:
            raise ValueError(f"Memory type '{self.type}' must be one of {MEMORY_TYPES}")
        if not (1 <= self.importance <= 10):
            raise ValueError(f"Importance must be 1–10, got {self.importance}")

    def to_dict(self) -> dict:
        # converts the Memory object to a plain Python dict for JSONL logging
        # the key step is .tolist() on the embedding —
        # numpy arrays are not JSON-serialisable, but Python lists are
        return {
            "timestamp": self.timestamp,
            "description": self.description,
            "importance": self.importance,
            "embedding": self.embedding.tolist(),  # numpy array → plain Python list
            "type": self.type,
        }

    def __repr__(self) -> str:
        # __repr__ controls what prints when you do print(memory) or inspect a Memory object
        # we override the default repr because the full embedding array
        # (384 floats) would make it unreadable — so we only show the first 60 chars
        return (
            f"Memory(day={self.timestamp}, type={self.type}, "
            f"importance={self.importance}, "
            f"desc='{self.description[:60]}...')"
        )


class MemoryStream:
    """
    Append-only memory stream for a single agent.
    Agents never forget — retrieval controls what surfaces at decision time.
    The stream grows throughout the simulation.
    """

    def __init__(self, agent_name: str):
        # store the agent's name so we can label output clearly
        self.agent_name = agent_name
        # the stream is just a Python list of Memory objects
        # underscore prefix (_memories) signals it's private — access via methods not directly
        self._memories: List[Memory] = []

    # ── Write ─────────────────────────────────────────────────────────────────

    def add(self, description, importance, embedding, memory_type, timestamp=0):
        # create a Memory object from the raw inputs and append it to the list
        # this is the only way to add memories — the stream is append-only
        # (we never delete or modify existing memories, only add new ones)
        memory = Memory(timestamp=timestamp, description=description,
                        importance=importance, embedding=embedding, type=memory_type)
        self._memories.append(memory)
        return memory

    # ── Read ──────────────────────────────────────────────────────────────────

    def get_all(self):
        # returns a copy of the full list in chronological order
        # used in Stage 1 where we pass all memories to the LLM
        # list() creates a copy so external code can't accidentally modify _memories
        return list(self._memories)

    def get_recent(self, n):
        # returns the n most recent memories using Python list slicing
        # [-n:] means 'start from n positions before the end'
        # e.g. get_recent(3) on a 7-memory stream returns memories 5, 6, 7
        return self._memories[-n:]

    def get_by_type(self, memory_type):
        # filters memories by type using a list comprehension
        # e.g. get_by_type('reflection') returns only reflection memories
        # useful in Stage 5 when we want to inspect what beliefs an agent has formed
        return [m for m in self._memories if m.type == memory_type]

    def get_cumulative_importance(self, since_index=0):
        # sums the importance scores of memories from since_index onwards
        # used by reflection.py in Stage 5 to decide when to trigger a reflection
        # e.g. if memories since the last reflection exceed a threshold, reflect now
        return sum(m.importance for m in self._memories[since_index:])

    def count(self):
        # simple helper — returns the total number of memories in the stream
        return len(self._memories)

    # ── Inspect ───────────────────────────────────────────────────────────────

    def pretty_print(self, n=None):
        # if n is provided, only print the last n memories
        # if n is None (default), print all memories
        memories = self._memories if n is None else self._memories[-n:]
        print(f"\n{'='*60}")
        print(f"Memory stream: {self.agent_name} ({self.count()} total memories)")
        print(f"{'='*60}")
        for i, m in enumerate(memories):
            # note: embedding is intentionally not printed —
            # it's 384 floats and would be completely unreadable
            print(f"\n[{i+1}] Day {m.timestamp} | {m.type.upper()} | Importance: {m.importance}/10")
            print(f"    {m.description}")
        print(f"{'='*60}\n")

print("✓ Memory and MemoryStream classes loaded.")

## 2. Embedding Model

Local sentence-transformers model (all-MiniLM-L6-v2). No API key needed — downloads once (~80MB) and runs on-device.

In [ ]:
%pip install sentence-transformers

# sentence-transformers: HuggingFace library that wraps pre-trained embedding models
# all-MiniLM-L6-v2: a lightweight model (~80MB) that converts text → 384-dim float vectors
# 'normalize_embeddings=True' ensures every vector has unit length (norm=1.0)
# which makes cosine similarity equivalent to a simple dot product — faster in Stage 2
from sentence_transformers import SentenceTransformer

# downloads once on first run, then cached locally — no API key or cost
model = SentenceTransformer("all-MiniLM-L6-v2")

def embed(text: str) -> np.ndarray:
    # encode() runs the transformer forward pass and returns a numpy array
    # this is called for every memory description when it is first created
    # and for every query in Stage 2 retrieval
    return model.encode(text, normalize_embeddings=True)

# Quick sanity check — norm should be exactly 1.0 due to normalize_embeddings=True
v = embed("hello world")
print(f"Shape: {v.shape}, norm: {np.linalg.norm(v):.4f}")

## 3. Test `Memory` Dataclass

### 3a. Valid construction

In [ ]:
# create a Memory object with a real embedding to verify valid construction works end-to-end
# in production this comes from embed() in client.py — here we call it directly
m = Memory(
    timestamp=1,
    description="Agent observed a wildfire approaching from the east.",
    importance=9,
    embedding=embed("wildfire approaching"),
    type="observation",
)

# __repr__ kicks in here — shows a readable summary without the full embedding array
print(m)

# to_dict() converts to a JSON-serialisable format for JSONL logging
# the key check: embedding is now a list (not numpy) and length is 384
print(f"\nto_dict keys: {list(m.to_dict().keys())}")
print(f"Embedding length in dict: {len(m.to_dict()['embedding'])}")

### 3b. Validation — invalid type

In [ ]:
# demonstrate __post_init__ validation — an invalid type should raise immediately
# 'hallucination' is not in MEMORY_TYPES so this must fail
# try/except lets us verify the error is raised without crashing the notebook
try:
    Memory(timestamp=0, description="bad", importance=5,
           embedding=embed("x"), type="hallucination")
except ValueError as e:
    print(f"✓ Caught: {e}")

### 3c. Validation — importance out of range

In [ ]:
# demonstrate importance validation — both ends of the range must be rejected
# importance=11 exceeds the maximum of 10
try:
    Memory(timestamp=0, description="bad", importance=11,
           embedding=embed("x"), type="observation")
except ValueError as e:
    print(f"✓ Caught: {e}")

# importance=0 falls below the minimum of 1
try:
    Memory(timestamp=0, description="bad", importance=0,
           embedding=embed("x"), type="observation")
except ValueError as e:
    print(f"✓ Caught: {e}")

## 4. Test `MemoryStream`

### 4a. Add memories

In [ ]:
# create an empty memory stream for a test agent
# in the real simulation each agent (Jennifer, Edward, etc.) gets their own stream
stream = MemoryStream(agent_name="Firefighter_01")

# test memories covering all four types across three simulation days
# (description, importance, memory_type, timestamp)
test_memories = [
    ("Received dispatch to Station 5.", 3, "observation", 0),
    ("Checked equipment inventory — low on hoses.", 5, "observation", 0),
    ("Decided to request additional resources from HQ.", 7, "decision", 1),
    ("Discussed evacuation zones with Coordinator_01.", 6, "conversation", 1),
    ("Reflected: resource shortages are a recurring pattern.", 8, "reflection", 2),
    ("Observed fire shifted direction to the north.", 9, "observation", 2),
    ("Decided to reposition crew to northern perimeter.", 8, "decision", 3),
]

# stream.add() creates a Memory object and appends it — the only write path
# embed() converts each description to a 384-dim vector for later retrieval
for desc, imp, mtype, day in test_memories:
    stream.add(desc, imp, embed(desc), mtype, timestamp=day)

print(f"Stream count: {stream.count()}")

### 4b. `get_all()`

In [ ]:
# get_all() returns every memory in chronological insertion order
# in Stage 1 we pass all memories to the LLM as the agent's full context
all_mems = stream.get_all()
print(f"Total: {len(all_mems)}")
for m in all_mems:
    print(f"  Day {m.timestamp} | {m.type:13s} | imp={m.importance} | {m.description[:50]}")

### 4c. `get_recent(n)`

In [ ]:
# get_recent(n) returns the n most recent memories using Python list slicing [-n:]
# useful for giving the LLM a short recency window without the full stream
recent = stream.get_recent(3)
print("Last 3 memories:")
for m in recent:
    print(f"  {m}")

### 4d. `get_by_type()`

In [ ]:
# get_by_type() filters using a list comprehension — returns only memories of that type
# in Stage 5 we use get_by_type('reflection') to inspect what beliefs an agent has formed
# here we loop over all types to confirm each is represented correctly
for mtype in sorted(MEMORY_TYPES):
    subset = stream.get_by_type(mtype)
    print(f"{mtype:13s}: {len(subset)} memories")

### 4e. `get_cumulative_importance()`

In [ ]:
# get_cumulative_importance() sums importance scores from a given index onwards
# reflection.py in Stage 5 calls this to decide whether to trigger a new reflection:
# if the cumulative importance since the last reflection exceeds a threshold, reflect now
total_imp = stream.get_cumulative_importance()
print(f"Total importance (all):        {total_imp}")

# since_index=4 means 'sum from memory 5 onwards' (0-indexed)
since_4 = stream.get_cumulative_importance(since_index=4)
print(f"Total importance (index 4+):   {since_4}")

# manual verification using get_all() — both paths should agree
manual = sum(m.importance for m in stream.get_all()[4:])
print(f"Manual check:                  {manual}")
assert since_4 == manual, "Mismatch!"
print("✓ Matches.")

### 4f. `pretty_print()`

In [ ]:
# pretty_print(n) prints the last n memories in a readable format
# embedding is intentionally omitted — 384 floats would be completely unreadable
# used during debugging to inspect what an agent 'remembers' at a given simulation step
stream.pretty_print(n=3)

## 5. Serialization Round-Trip

In [ ]:
import json

# serialize all memories to JSONL — one JSON object per line
# this is the format used by logger.py to persist the stream to disk after each day
jsonl_lines = [json.dumps(m.to_dict()) for m in stream.get_all()]
print(f"Serialized {len(jsonl_lines)} memories to JSONL")
print(f"\nSample line:\n{jsonl_lines[0][:120]}...")

# deserialize one back to verify the round-trip works correctly
# json.loads() returns a plain dict — we reconstruct the Memory manually
# the critical step: np.array(loaded["embedding"]) converts the list back to a numpy array
loaded = json.loads(jsonl_lines[0])
restored = Memory(
    timestamp=loaded["timestamp"],
    description=loaded["description"],
    importance=loaded["importance"],
    embedding=np.array(loaded["embedding"]),  # list → numpy array
    type=loaded["type"],
)
print(f"\nRestored: {restored}")
print("✓ Round-trip works.")

## 6. Embedding Similarity Demo

Preview of how retrieval.py would use these embeddings.

In [ ]:
def cosine_sim(a, b):
    # cosine similarity measures the angle between two vectors, not their magnitude
    # because embeddings are normalised (norm=1.0), np.dot(a, b) is already cosine similarity
    # but we use the full formula here for clarity and correctness with any vectors
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))

# embed the retrieval query — in Stage 2 this would be the current situation description
query_emb = embed("fire changed direction")

# rank all memories by similarity to the query, highest first
# this is the core of the retrieval step in Park et al. —
# the most semantically similar memories surface at decision time
print("Similarity to query: 'fire changed direction'\n")
ranked = sorted(stream.get_all(), key=lambda m: cosine_sim(m.embedding, query_emb), reverse=True)
for m in ranked:
    sim = cosine_sim(m.embedding, query_emb)
    print(f"  {sim:+.4f}  | {m.description[:55]}")

## 7. Reflection Trigger Check

Park et al. trigger reflection when cumulative importance since last reflection exceeds a threshold (e.g. 50).

In [ ]:
# Park et al. trigger reflection when cumulative importance since the last reflection
# exceeds a threshold — the agent pauses to synthesise higher-level beliefs
# this avoids reflecting after every single memory (too frequent) or never (too sparse)
REFLECTION_THRESHOLD = 25

# last_reflection_idx tracks where in the stream the last reflection occurred
# in reflection.py this is updated each time a reflection is written
last_reflection_idx = 0  # 0 = no reflection has happened yet

cum = stream.get_cumulative_importance(since_index=last_reflection_idx)
print(f"Cumulative importance since index {last_reflection_idx}: {cum}")
print(f"Threshold: {REFLECTION_THRESHOLD}")
print(f"Trigger reflection? {'YES ✓' if cum >= REFLECTION_THRESHOLD else 'No'}")